# SupeRR Selector Season 2 — Rajasthan Royals IPL 2027 Auction Analytics
## Identifying the Most Reliable Players Under Pressure
**Participant / Candidate:** Prokash Ghosh  
**Competition:** SupeRR Selector Season 2 (Round 1 Official Submission)  
**Franchise:** Rajasthan Royals (via Unstop Platform)  

---
### Overview
This notebook contains the complete, reproducible analytical workflow developed for Round 1 of the SupeRR Selector Season 2 competition organized by Rajasthan Royals on Unstop.

### Table of Contents
1. **Data Ingestion & Integrity Audit**
2. **Situational Context & Pressure State Modeling**
3. **League-Wide Over & Phase Baselines**
4. **Batting Pressure Reliability Index (BPRI)**
5. **Bowling Pressure Reliability Index (BoPRI)**
6. **Multi-Model Sensitivity Analysis**
7. **Rajasthan Royals Strategic Auction Target Shortlist**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# Set display options
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)
print('Libraries imported successfully.')

## 1. Load Data Pack

In [ ]:
data_dir = 'Data Pack'
bbb = pd.read_csv(os.path.join(data_dir, 'ball_by_ball.csv'), low_memory=False)
matches = pd.read_csv(os.path.join(data_dir, 'matches.csv'))
comps = pd.read_csv(os.path.join(data_dir, 'competitions.csv'))
ctpm = pd.read_csv(os.path.join(data_dir, 'competition_team_player_mapping.csv'))

print(f'Ball by ball records: {len(bbb):,}')
print(f'Total matches: {len(matches):,}')
print(f'Total competitions: {len(comps):,}')

## 2. Situational Reconstruction & Pressure Definition

In [ ]:
# Chronological sort
bbb = bbb.sort_values(['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num']).reset_index(drop=True)

# Ball metrics
bbb['is_legal_ball'] = ~bbb['extras_type'].isin(['wide', 'Overstep'])
bbb['is_batter_ball'] = bbb['extras_type'] != 'wide'
bbb['ball_total_runs'] = bbb['runs'] + bbb['extras1_runs'] + bbb['extras_2_runs']
bbb['is_bowler_extra'] = bbb['extras_type'].isin(['wide', 'Overstep', 'other'])
bbb['bowler_runs'] = bbb['runs'] + np.where(bbb['is_bowler_extra'], bbb['extras1_runs'] + bbb['extras_2_runs'], 0)
bowler_dismissals = ['Bowled', 'Caught', 'LBW', 'Stump', 'Hit Wicket']
bbb['is_bowler_wicket'] = bbb['dismissal_type'].isin(bowler_dismissals)

# State reconstruction
bbb['cum_runs'] = bbb.groupby(['match_id', 'innings'])['ball_total_runs'].cumsum() - bbb['ball_total_runs']
bbb['cum_wickets'] = bbb.groupby(['match_id', 'innings'])['is_wicket'].cumsum() - bbb['is_wicket'].astype(int)
bbb['legal_balls_bowled'] = bbb.groupby(['match_id', 'innings'])['is_legal_ball'].cumsum() - bbb['is_legal_ball'].astype(int)
bbb['balls_remaining'] = np.maximum(0, 120 - bbb['legal_balls_bowled'])
bbb['wickets_in_hand'] = 10 - bbb['cum_wickets']

# Targets
inn1_totals = bbb[bbb['innings'] == 1].groupby('match_id')['ball_total_runs'].sum().reset_index()
inn1_totals.rename(columns={'ball_total_runs': 'inn1_total'}, inplace=True)
matches_merged = matches[['match_id', 'comp_id', 'revised_target', 'knock_out', 'stage']].merge(
    comps[['comp_id', 'comp_name', 'match_format', 'season']], on='comp_id', how='left'
)
bbb = bbb.merge(inn1_totals, on='match_id', how='left')
bbb = bbb.merge(matches_merged, on='match_id', how='left')

bbb['target'] = np.where(bbb['revised_target'].notnull() & (bbb['revised_target'] > 0),
                         bbb['revised_target'],
                         bbb['inn1_total'] + 1)
bbb['runs_needed'] = np.maximum(0, bbb['target'] - bbb['cum_runs'])
bbb['rrr'] = np.where(
    (bbb['innings'] == 2) & (bbb['balls_remaining'] > 0),
    (bbb['runs_needed'] * 6.0) / bbb['balls_remaining'],
    0.0
)

# Over baselines
over_baselines = bbb.groupby('overid').agg(
    exp_runs=('bowler_runs', 'mean'),
    exp_wicket=('is_bowler_wicket', 'mean')
).reset_index()
bbb = bbb.merge(over_baselines, on='overid', how='left')

# Pressure definition
p_super_over = bbb['innings'] > 2
p_chase_death = (bbb['innings'] == 2) & (bbb['overid'] >= 16) & (bbb['runs_needed'] > 0) & (bbb['runs_needed'] <= 75) & (bbb['rrr'] >= 7.0) & (bbb['rrr'] <= 24.0)
p_chase_crunch = (bbb['innings'] == 2) & (bbb['overid'].between(11, 15)) & (bbb['rrr'] >= 9.5) & (bbb['rrr'] <= 18.0) & (bbb['wickets_in_hand'] <= 7)
p_chase_close = (bbb['innings'] == 2) & (bbb['balls_remaining'] <= 36) & (bbb['runs_needed'] > 0) & (bbb['runs_needed'] <= 55) & (bbb['rrr'] >= 7.0)
p_chase_collapse = (bbb['innings'] == 2) & (bbb['cum_wickets'] >= 4) & (bbb['runs_needed'] >= 30) & (bbb['balls_remaining'] >= 12) & (bbb['rrr'] <= 20.0)
p_inn1_death = (bbb['innings'] == 1) & (bbb['overid'] >= 16)
p_inn1_collapse = (bbb['innings'] == 1) & (bbb['overid'] <= 10) & (bbb['cum_wickets'] >= 3)

bbb['is_pressure'] = p_super_over | p_chase_death | p_chase_crunch | p_chase_close | p_chase_collapse | p_inn1_death | p_inn1_collapse
print(f'Pressure deliveries identified: {bbb["is_pressure"].sum():,} ({bbb["is_pressure"].mean()*100:.1f}%)')

## 3. Top 10 Batters Under Pressure (BPRI)

In [ ]:
bat_top10 = pd.read_csv('rankings/top_10_batters_pressure.csv')
cols = ['rank', 'striker_name', 'primary_role', 'bf_p', 'runs_p', 'sr_p', 'avg_p', 'sr_delta', 'consistency_rate', 'BPRI']
bat_top10[cols]

## 4. Top 10 Bowlers Under Pressure (BoPRI)

In [ ]:
bowl_top10 = pd.read_csv('rankings/top_10_bowlers_pressure.csv')
cols_b = ['rank', 'bowler_name', 'primary_role', 'balls_p', 'runs_p', 'wickets_p', 'econ_p', 'sr_p', 'overs_saved_p', 'econ_delta', 'BoPRI']
bowl_top10[cols_b]

## 5. Rajasthan Royals IPL 2027 Auction Shortlist

In [ ]:
rr_targets = pd.read_csv('rankings/rajasthan_royals_auction_shortlist.csv')
rr_targets